In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             balanced_accuracy_score, confusion_matrix,
                             matthews_corrcoef, cohen_kappa_score)
from sklearn.base import BaseEstimator, ClassifierMixin
import time
import os

# Set random seed for reproducibility
np.random.seed(42)

# Set the number of K folds as a global variable
K_FOLDS = 5

# FHO Algorithm Implementation for Classification
class FHOClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, num_bees=50, num_iterations=100, r1=0.5, r2=0.5, r3=0.5, lower_bound=-10, upper_bound=10):
        self.num_bees = num_bees
        self.num_iterations = num_iterations
        self.r1 = r1
        self.r2 = r2
        self.r3 = r3
        self.lower_bound = lower_bound
        self.upper_bound = upper_bound
        self.best_solution = None
        self.best_fitness = float('-inf')
        self.classes_ = None
        self.n_classes_ = None
        
    def _initialize_population(self, n_features, n_classes):
        """Initialize the honeybee population"""
        # Each bee is represented as a weight matrix for features and classes
        population = np.random.uniform(
            self.lower_bound, 
            self.upper_bound, 
            (self.num_bees, n_features, n_classes)
        )
        return population
    
    def _calculate_fitness(self, bee, X, y):
        """Calculate fitness of a bee based on classification accuracy"""
        # Predict using the bee's weight matrix
        y_pred = self._predict_proba_bee(bee, X)
        y_pred = np.argmax(y_pred, axis=1)
        
        # Convert to class labels
        y_pred_labels = np.array([self.classes_[i] for i in y_pred])
        
        # Calculate accuracy
        accuracy = accuracy_score(y, y_pred_labels)
        return accuracy
    
    def _predict_proba_bee(self, bee, X):
        """Calculate probability outputs for each class using a bee's weights"""
        # Dot product of features and bee weights
        raw_output = np.dot(X, bee)
        
        # Apply softmax to get probabilities
        exp_output = np.exp(raw_output - np.max(raw_output, axis=1, keepdims=True))
        probabilities = exp_output / np.sum(exp_output, axis=1, keepdims=True)
        
        return probabilities
    
    def _update_position(self, population, fitness):
        """Update the position of bees based on FHO algorithm"""
        best_bee_idx = np.argmax(fitness)
        best_bee = population[best_bee_idx].copy()
        
        new_population = population.copy()
        
        for i in range(self.num_bees):
            if i != best_bee_idx:
                # FHO update equation
                r1 = np.random.random()
                r2 = np.random.random()
                r3 = np.random.random()
                
                # Select a random bee (different from current)
                random_bee_idx = np.random.choice([j for j in range(self.num_bees) if j != i])
                random_bee = population[random_bee_idx]
                
                # Update position
                new_position = population[i] + r1 * (best_bee - population[i]) + \
                               r2 * (random_bee - population[i]) + \
                               r3 * (np.random.random(population[i].shape) * 2 - 1)
                
                # Clip to bounds
                new_position = np.clip(new_position, self.lower_bound, self.upper_bound)
                new_population[i] = new_position
        
        return new_population
    
    def fit(self, X, y):
        """Fit the FHO classifier to the training data"""
        # Encode the target labels
        self.label_encoder_ = LabelEncoder()
        y_encoded = self.label_encoder_.fit_transform(y)
        self.classes_ = self.label_encoder_.classes_
        self.n_classes_ = len(self.classes_)
        
        n_samples, n_features = X.shape
        
        # Initialize population
        population = self._initialize_population(n_features, self.n_classes_)
        
        # FHO algorithm iterations
        for iteration in range(self.num_iterations):
            # Calculate fitness for each bee
            fitness = np.array([self._calculate_fitness(bee, X, y) for bee in population])
            
            # Update best solution
            best_idx = np.argmax(fitness)
            if fitness[best_idx] > self.best_fitness:
                self.best_fitness = fitness[best_idx]
                self.best_solution = population[best_idx].copy()
            
            # Update positions
            population = self._update_position(population, fitness)
            
            # Print progress
            if iteration % 10 == 0:
                print(f"Iteration {iteration}, Best Fitness: {self.best_fitness}")
        
        return self
    
    def predict_proba(self, X):
        """Predict class probabilities for X"""
        if self.best_solution is None:
            raise ValueError("Model not fitted yet.")
        
        return self._predict_proba_bee(self.best_solution, X)
    
    def predict(self, X):
        """Predict class labels for X"""
        if self.best_solution is None:
            raise ValueError("Model not fitted yet.")
        
        proba = self.predict_proba(X)
        predictions = np.argmax(proba, axis=1)
        return np.array([self.classes_[i] for i in predictions])

# Load and prepare the dataset
# Updated path to the new dataset
data_path = "../datasets/dataset-2.csv"  
df = pd.read_csv(data_path)

# Exploratory data analysis
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Class distribution:")
print(df['label'].value_counts())

# Prepare the features and target
X = df.drop(['label'], axis=1)
y = df['label']

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Create directories for saving results
os.makedirs("results/dataset2", exist_ok=True)
os.makedirs("results/dataset2/confusion_matrices", exist_ok=True)

# Initialize results storage
results = []
timing_results = []

# K-fold cross-validation
kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

fold_idx = 1
for train_index, test_index in kf.split(X_scaled):
    print(f"\nProcessing fold {fold_idx}...")
    
    # Split the data
    X_train, X_test = X_scaled[train_index], X_scaled[test_index]
    y_train, y_test = y.iloc[train_index].values, y.iloc[test_index].values
    
    # Train the FHO classifier
    clf = FHOClassifier(num_bees=30, num_iterations=50)
    
    start_train_time = time.time()
    clf.fit(X_train, y_train)
    train_time = time.time() - start_train_time
    
    start_test_time = time.time()
    y_pred = clf.predict(X_test)
    test_time = time.time() - start_test_time
    
    # Record timing results
    timing_results.append({
        'Classifier': 'FHO',
        'Fold': fold_idx,
        'Training Time (s)': train_time,
        'Testing Time (s)': test_time,
        'Total Time (s)': train_time + test_time
    })
    
    # Calculate metrics
    unique_classes = np.unique(y)
    cm = confusion_matrix(y_test, y_pred, labels=unique_classes)
    
    # Plot and save confusion matrix
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=unique_classes, yticklabels=unique_classes)
    plt.title(f"FHO Classifier - Dataset 2 - Fold {fold_idx} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.savefig(f"results/dataset2/confusion_matrices/FHO_fold_{fold_idx}.png")
    plt.close()
    
    # Helper function for confusion matrix metrics
    def confusion_matrix_metrics(cm, classes):
        class_metrics = {}
        for idx, class_label in enumerate(classes):
            true_class = np.zeros(len(classes))
            true_class[idx] = 1
            
            TP = cm[idx, idx]
            FP = np.sum(cm[:, idx]) - TP
            FN = np.sum(cm[idx, :]) - TP
            TN = np.sum(cm) - (TP + FP + FN)
            
            class_metrics[class_label] = {
                'TPR': TP / (TP + FN) if (TP + FN) > 0 else 0,
                'TNR': TN / (TN + FP) if (TN + FP) > 0 else 0,
                'FPR': FP / (FP + TN) if (FP + TN) > 0 else 0,
                'FNR': FN / (FN + TP) if (FN + TP) > 0 else 0
            }
        return class_metrics
    
    # Calculate per-class metrics
    cm_metrics = confusion_matrix_metrics(cm, unique_classes)
    
    # Calculate overall metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    balanced_acc = balanced_accuracy_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    
    # Store overall metrics
    overall_metrics = {
        'Classifier': 'FHO',
        'Fold': fold_idx,
        'Class': 'Overall',
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1,
        'Balanced Accuracy': balanced_acc,
        'MCC': mcc,
        'Kappa': kappa,
        'Training Time (s)': train_time,
        'Testing Time (s)': test_time
    }
    results.append(overall_metrics)
    
    # Store class-specific metrics
    for class_label in unique_classes:
        class_mask = (y_test == class_label)
        if np.any(class_mask):
            class_metrics = {
                'Classifier': 'FHO',
                'Fold': fold_idx,
                'Class': class_label,
                'Accuracy': accuracy_score(y_test == class_label, y_pred == class_label),
                'Precision': precision_score(y_test == class_label, y_pred == class_label, average='binary'),
                'Recall': recall_score(y_test == class_label, y_pred == class_label, average='binary'),
                'F1 Score': f1_score(y_test == class_label, y_pred == class_label, average='binary'),
                'TPR': cm_metrics[class_label]['TPR'],
                'TNR': cm_metrics[class_label]['TNR'],
                'FPR': cm_metrics[class_label]['FPR'],
                'FNR': cm_metrics[class_label]['FNR'],
                'Training Time (s)': train_time,
                'Testing Time (s)': test_time
            }
            results.append(class_metrics)
    
    fold_idx += 1

# Save results to CSV
results_df = pd.DataFrame(results)
results_df.to_csv("results/dataset2/metrics.csv", index=False)

timing_df = pd.DataFrame(timing_results)
timing_df.to_csv("results/dataset2/timing.csv", index=False)

# Calculate and print average metrics across folds
avg_metrics = results_df[results_df['Class'] == 'Overall'].mean(numeric_only=True)
print("\nAverage Metrics Across All Folds:")
print(avg_metrics)

# Plot performance metrics
plt.figure(figsize=(12, 8))
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'Balanced Accuracy']
fold_metrics = results_df[results_df['Class'] == 'Overall'][['Fold'] + metrics_to_plot]

fold_metrics_melted = pd.melt(fold_metrics, id_vars=['Fold'], value_vars=metrics_to_plot)
sns.barplot(data=fold_metrics_melted, x='Fold', y='value', hue='variable')
plt.title('FHO Classifier Performance Across Folds - Dataset 2')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.savefig('results/dataset2/performance_metrics.png')
plt.close()

# Plot timing results
plt.figure(figsize=(10, 6))
sns.barplot(data=timing_df, x='Fold', y='Total Time (s)')
plt.title('FHO Classifier - Total Runtime Per Fold - Dataset 2')
plt.savefig('results/dataset2/timing_plot.png')
plt.close()

# Report summary
print("\nClassification Complete!")
print(f"Results saved to results/dataset2/metrics.csv")
print(f"Timing information saved to results/dataset2/timing.csv")
print(f"Confusion matrices saved to results/dataset2/confusion_matrices/")
print(f"Performance plots saved to results/dataset2/")

FileNotFoundError: [Errno 2] No such file or directory: '../datasets/dataset-2.csv'